In [ ]:
import json
import xml.etree.ElementTree as ET
from pathlib import Path

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
xml_file = "Base_Model_Arch_vs_Structural_Clashes.xml"
output_json = "clashes_normalized.json"
usable_output_json = "usable_clashes.json"


# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------
def text_or_none(node, path):
    child = node.find(path)
    return child.text.strip() if child is not None and child.text else None


def to_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def parse_name_value_nodes(parent, path):
    out = {}
    for item in parent.findall(path):
        name = text_or_none(item, "name")
        value = text_or_none(item, "value")
        if name:
            out[name] = value
    return out


def extract_created_at(clashresult):
    date_node = clashresult.find("./createddate/date")
    if date_node is None:
        return None

    try:
        return (
            f"{int(date_node.attrib.get('year')):04d}-"
            f"{int(date_node.attrib.get('month')):02d}-"
            f"{int(date_node.attrib.get('day')):02d}T"
            f"{int(date_node.attrib.get('hour', 0)):02d}:"
            f"{int(date_node.attrib.get('minute', 0)):02d}:"
            f"{int(date_node.attrib.get('second', 0)):02d}"
        )
    except Exception:
        return None


def extract_clash_point(clashresult):
    pt = clashresult.find("./clashpoint/pos3f")
    if pt is None:
        return None

    return {
        "x": to_float(pt.attrib.get("x")),
        "y": to_float(pt.attrib.get("y")),
        "z": to_float(pt.attrib.get("z")),
    }


def extract_revit_global_id(attrs):
    candidate_keys = [
        "Revit UniqueId",
        "Revit Unique ID",
        "UniqueId",
        "Unique ID",
        "GlobalId",
        "Global ID",
        "IfcGUID",
        "Ifc GUID",
    ]
    for key in candidate_keys:
        if key in attrs and attrs[key]:
            return attrs[key]
    return None

# ------------------------------------------------------------
# CORE PARSERS
# ------------------------------------------------------------
def parse_clash_object(clashobject):
    attrs = parse_name_value_nodes(clashobject, "./objectattribute")
    tags = parse_name_value_nodes(clashobject, "./smarttags/smarttag")

    base_obj = {
        "revitGlobalId": extract_revit_global_id(attrs),
        "elementId": attrs.get("Element ID"),
        "clashMetadata": {
            "layer": text_or_none(clashobject, "./layer"),
            "itemName": tags.get("Item Name"),
            "itemType": tags.get("Item Type"),
        },
        "rawAttributes": attrs,
        "rawSmartTags": tags,
    }

    return base_obj


def parse_clash_result(clashresult):
    return {
        "clashName": clashresult.attrib.get("name"),
        "clashGuid": clashresult.attrib.get("guid"),
        "clashMetadata": {
            "status": clashresult.attrib.get("status"),
            "description": text_or_none(clashresult, "./description"),
            "resultStatus": text_or_none(clashresult, "./resultstatus"),
            "distance": to_float(clashresult.attrib.get("distance")),
            "href": clashresult.attrib.get("href"),
            "gridLocation": text_or_none(clashresult, "./gridlocation"),
            "createdAt": extract_created_at(clashresult),
            "clashPoint": extract_clash_point(clashresult),
        },
        "objects": [
            parse_clash_object(obj)
            for obj in clashresult.findall("./clashobjects/clashobject")
        ],
    }


def parse_clash_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    output = {
        "sourceFile": Path(xml_path).name,
        "sourcePath": str(Path(xml_path).resolve()),
        "units": root.attrib.get("units"),
        "tests": []
    }

    for clashtest in root.findall(".//clashtest"):
        summary = clashtest.find("./summary")

        test_data = {
            "testName": clashtest.attrib.get("name"),
            "testType": clashtest.attrib.get("test_type"),
            "testStatus": clashtest.attrib.get("status"),
            "tolerance": clashtest.attrib.get("tolerance"),
            "summary": {
                "total": int(summary.attrib["total"]) if summary is not None and summary.attrib.get("total") else None,
                "new": int(summary.attrib["new"]) if summary is not None and summary.attrib.get("new") else None,
                "active": int(summary.attrib["active"]) if summary is not None and summary.attrib.get("active") else None,
                "reviewed": int(summary.attrib["reviewed"]) if summary is not None and summary.attrib.get("reviewed") else None,
                "approved": int(summary.attrib["approved"]) if summary is not None and summary.attrib.get("approved") else None,
                "resolved": int(summary.attrib["resolved"]) if summary is not None and summary.attrib.get("resolved") else None,
            },
            "clashes": []
        }

        for clashresult in clashtest.findall("./clashresults/clashresult"):
            test_data["clashes"].append(parse_clash_result(clashresult))

        output["tests"].append(test_data)
    
    return output

def optimize_clash_for_agent(clash):
    """
    Minifies a BIM clash object for LLM token optimization.
    Extracts only the ID, clash type, and core item metadata.
    """
    return {
        "id": clash.get("clashGuid"),
        "type": clash.get("clashMetadata", {}).get("description"),
        "items": [
            {
                "n": obj.get("clashMetadata", {}).get("itemName"),
                "t": obj.get("clashMetadata", {}).get("itemType")
            } 
            for obj in clash.get("objects", [])
        ]
    }

# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------
if __name__ == "__main__":
    data = parse_clash_xml(xml_file)
    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    raw_clashes = data["tests"][0]["clashes"]

## Infer severity using LLM

In [ ]:
PREPROMPT = """
Role: You are a BIM Coordination Engine. Your mission is to analyze multi-object BIM clashes and provide a strategic resolution plan by identifying the "Lead" (the element that stays) and the "Movers" (the elements that must reroute).

1. Trade Priority Hierarchy (High to Low):

STR (Structural): Beams, Columns, Slabs, Metal Decks.

PLUMB-G (Gravity Plumbing): Sanitary, Storm, Drainage.

MECH-L (Large Mechanical): Primary Duct Mains, AHUs.

PLUMB-P (Pressurized): Domestic Water, Gas, Fire Protection.

ELEC (Electrical): Cable Trays, Conduits, Lighting.

2. Triage Logic:

Determine Severity: * CRITICAL: Any clash containing 2 or more items from Priority 1 or 2.

MEDIUM: Any clash containing Priority 3 vs. Priority 1/2.

LOW: Clashes containing only Priority 4 and 5 items.

Identify the "Lead": The item with the highest priority in the hierarchy. If there is a tie between two items of the same priority (e.g., two structural beams), both are designated as "Lead."

3. Discipline Codes:
ARC (Architectural), STR (Structural), MECH (Mechanical), PLUMB (Plumbing), FP (Fire Protection), ELEC (Electrical).

Input Format:

JSON
{
  "id": "uuid",
  "type": "Hard | Soft",
  "items": [{ "n": "Name", "t": "Type" }, ...]
}
Output Format:
Return a JSON array of objects:

JSON
[
  {
    "clash": "uuid",
    "severity": "LOW | MEDIUM | CRITICAL",
    "disciplines": ["CODE1", "CODE2"],
    "lead": ["Item Name 1"],
  }
]
"""

In [ ]:
import os
import re
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from llama_index.core.llms import ChatMessage
from llama_index.core.base.llms.types import MessageRole
from llama_index.llms.openai import OpenAI

for _env_path in (
    Path.cwd() / ".env",
    Path.cwd().parent / ".env",
    Path.cwd() / "apps" / "api" / ".env",
):
    if _env_path.is_file():
        load_dotenv(_env_path)
        break
else:
    load_dotenv()

def batch_clashes(clashes, max_batch_size=50, max_chars=None):
    """
    Batches a list of optimized clash objects.
    
    Args:
        clashes (list): The list of optimized clash dictionaries.
        max_batch_size (int): Max number of clashes per batch.
        max_chars (int): Optional character limit per batch (approximate token safety).
        
    Returns:
        list: A list of lists, where each sub-list is a batch.
    """
    batches = []
    current_batch = []
    current_chars = 0

    for clash in clashes:
        # Calculate size of the current clash in characters
        clash_json = json.dumps(clash)
        clash_len = len(clash_json)

        # Logic for token-aware batching (if max_chars is set)
        if max_chars and (current_chars + clash_len > max_chars):
            batches.append(current_batch)
            current_batch = []
            current_chars = 0
            
        current_batch.append(clash)
        current_chars += clash_len

        # Logic for count-based batching
        if not max_chars and len(current_batch) >= max_batch_size:
            batches.append(current_batch)
            current_batch = []

    # Catch the remaining items
    if current_batch:
        batches.append(current_batch)
        
    return batches

def _strip_code_fence(text: str) -> str:
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
        text = re.sub(r"\s*```\s*$", "", text)
    return text


def _clash_payload(clash: dict[str, Any], *, minify: bool) -> dict[str, Any]:
    if minify and "clashGuid" in clash:
        return optimize_clash_for_agent(clash)
    return clash


def infer_clash_severities(
    clashes: list[dict[str, Any]],
    *,
    model: str,
    api_key: str | None = None,
    preprompt: str = PREPROMPT,
    api_base: str | None = None,
    batch_size: int = 40,
    max_chars: int | None = None,
    minify: bool = True,
    temperature: float = 0.0,
) -> list[dict[str, Any]]:
    """Run batched LLM inference; each batch is one completion (system preprompt + user JSON)."""
    key = api_key or os.environ.get("OPENAI_API_KEY")
    if not key:
        raise ValueError("api_key is required (or set OPENAI_API_KEY).")

    llm = OpenAI(
        model=model,
        api_key=key,
        api_base=api_base,
        temperature=temperature,
    )

    optimized = [_clash_payload(c, minify=minify) for c in clashes]
    batches = batch_clashes(
        optimized, max_batch_size=batch_size, max_chars=max_chars
        )
    out: list[dict[str, Any]] = []
    for chunk in batches:
        user_content = json.dumps(chunk, ensure_ascii=False)
        messages = [
            ChatMessage(role=MessageRole.SYSTEM, content=PREPROMPT),
            ChatMessage(role=MessageRole.USER, content=user_content),
        ]
        resp = llm.chat(messages)
        text = _strip_code_fence(resp.message.content or "")
        print(text)
        parsed = json.loads(text)
        if not isinstance(parsed, list):
            raise ValueError(f"Expected JSON array from model, got {type(parsed)}")
        out.extend(parsed)
    return out

In [ ]:
import time

test_clashes = raw_clashes[:120]
t0 = time.perf_counter()
# severities = infer_clash_severities(
#     test_clashes,
#     # model=os.environ["MODEL_NAME"],
#     # api_base=os.environ.get("OPENAI_BASE_URL") or None,
#     model="gpt-4o-mini",
#     batch_size=40,
# )
elapsed = time.perf_counter() - t0
print(
    f"infer_clash_severities: {elapsed:.2f}s wall time "
    f"({len(test_clashes)} clashes, {len(severities)} results)"
)
print(severities)

In [ ]:
# Example (after load_dotenv cell): MODEL_NAME, OPENAI_API_KEY, OPENAI_BASE_URL from .env
# severities = infer_clash_severities(
#     usable_clashes[:120],
#     model=os.environ["MODEL_NAME"],
#     api_base=os.environ.get("OPENAI_BASE_URL") or None,
#     batch_size=40,
# )
